# Deepfake Detection — Image-Processing Track
**Scope:** face-swap + reenactment deepfakes, with and without screen-replay ("played to camera") recapture.
**Data:** FaceForensics++ (c23, Kaggle mirror) — official train/val/test splits, sub-sampled so it fits free Colab.

**How to use:** Runtime → Change runtime type → **T4 GPU**. Then run cells top to bottom.
Cells marked **🔧 YOU** need one action from you (paste token / upload file). Everything else is automatic.
Every stage saves to Google Drive, so if Colab disconnects you can re-run from the top and it resumes.

| Stage | Time (free T4, estimate) |
|---|---|
| 0–2 setup + Kaggle download + extract | 15–40 min (download speed dependent) |
| 3 face-crop clips | 5–10 min |
| 4 recapture simulation | 5–10 min |
| 5 features | 5 min |
| 6 classical experiments | 2 min |
| 7 CNN experiments | 10–15 min |
| 8 report | 1 min |

If Colab ever shows a **"Restart session"** popup after the install cell: click it, then re-run from the top.
If Colab disconnects mid-way: reconnect, re-run from the top — finished stages are restored from Drive instead of redone.

## Stage 0 — Setup

In [ ]:
import os, sys
import torch
if not torch.cuda.is_available():
    raise SystemExit("❌ No GPU. Menu: Runtime → Change runtime type → T4 GPU → Save. Then re-run from the top.")
print("GPU:", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount("/content/drive")
!pip -q install kaggle timm
!pip -q install facenet-pytorch --no-deps
print("✅ setup done")

### Config (don't change unless something runs out of time/disk)

In [ ]:
import os, sys, json, time, random, zipfile, shutil, glob, math, zlib, warnings, urllib.request
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
cv2.setNumThreads(2)

CFG = dict(
    KAGGLE_SLUG   = "xdxd003/ff-c23",
    WORK          = "/content/work",                          # local scratch (lost on disconnect)
    DRIVE         = "/content/drive/MyDrive/deepfake_project", # saved results (kept)
    PAIRS         = {"train": 60, "val": 15, "test": 30},     # official-split pairs used. 1 pair = 2 real + 2 fakes per method
    METHODS       = ["Deepfakes", "FaceSwap", "Face2Face", "NeuralTextures"],
    UNSEEN_METHOD = "FaceShifter",                            # never trained on -> "unknown attack" test
    T             = 8,        # frames per clip (consecutive)
    SIZE          = 224,      # face-crop size
    CLIPS_PER_VIDEO = 2,
    SEED          = 42,
    NJOBS         = max(1, os.cpu_count() or 1),
)
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"])

def P(*a):
    """path under WORK"""
    return os.path.join(CFG["WORK"], *a)
def D(*a):
    """path under Drive"""
    return os.path.join(CFG["DRIVE"], *a)

for d in [P("dl"), P("videos"), P("clips"), P("models"), D("results"), D("checkpoints")]:
    os.makedirs(d, exist_ok=True)
HAVE_CLIPS = os.path.exists(D("checkpoints", "clips_clean.tar"))   # True on a resumed run -> no re-download needed
print("work:", CFG["WORK"], "| drive:", CFG["DRIVE"], "| resuming from Drive checkpoint:", HAVE_CLIPS)

## Stage 1 — Get the dataset from Kaggle
**🔧 YOU (one-time, ~2 min):**
1. Make a free account at https://www.kaggle.com
2. Kaggle → profile picture → **Settings** → scroll to **API** → **Create New Token** (or *Generate New Token*).
3. Run the next cell. Either paste the token string it asks for, **or** if Kaggle downloaded a `kaggle.json` file, press Enter on the blank prompt and upload that file.

Dataset used: https://www.kaggle.com/datasets/xdxd003/ff-c23 (FaceForensics++, c23 compression: real + DeepFakes, FaceSwap, Face2Face, NeuralTextures, FaceShifter).
*Backup if that dataset ever disappears:* https://www.kaggle.com/datasets/adham7elmy/faceforencispp-extracted-frames (frames only, no video → temporal features would be lost; tell me and I'll adapt the notebook).

In [ ]:
import getpass, pathlib
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
tok = "" if HAVE_CLIPS else getpass.getpass("Paste Kaggle API token (or just press Enter to upload kaggle.json instead): ").strip()
if HAVE_CLIPS:
    print("Drive checkpoint found -> Kaggle not needed, skipping.")
elif tok:
    os.environ["KAGGLE_API_TOKEN"] = tok
    p = pathlib.Path("~/.kaggle/access_token").expanduser(); p.write_text(tok); p.chmod(0o600)
else:
    from google.colab import files
    up = files.upload()
    src = list(up)[0]
    dst = os.path.expanduser("~/.kaggle/kaggle.json")
    shutil.move(src, dst); os.chmod(dst, 0o600)
KS = CFG["KAGGLE_SLUG"]
if not HAVE_CLIPS:
    !kaggle datasets files {KS} | head -8
    print("↑ if you see a file list, Kaggle login works. If you see 401/403, redo the token step.")

In [ ]:
WORKDL = P("dl")
if HAVE_CLIPS:
    print("Drive checkpoint found -> skipping 10+ GB download.")
else:
    if not glob.glob(WORKDL + "/*.zip"):
        !kaggle datasets download -d {KS} -p {WORKDL}
    if not glob.glob(WORKDL + "/*.zip"):
        raise SystemExit("❌ Download failed. Open https://www.kaggle.com/datasets/xdxd003/ff-c23 in your browser while logged in, "
                         "check it exists, redo the token step, and run this cell again. Paste the error text to Claude if it persists.")
    ZIP = glob.glob(WORKDL + "/*.zip")[0]
    print("zip:", ZIP, round(os.path.getsize(ZIP) / 1e9, 1), "GB")
    !df -h /content | tail -1

## Stage 2 — Pick the subset (official FF++ splits) and extract only those videos

In [ ]:
SPLIT_URL = "https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/splits/{}.json"
CAT_KEYS  = {"deepfakes": "Deepfakes", "faceswap": "FaceSwap", "face2face": "Face2Face",
             "neuraltextures": "NeuralTextures", "faceshifter": "FaceShifter"}
REAL_DIRS = {"original", "original_sequences", "real", "originals", "pristine", "youtube"}

def load_pairs():
    pairs = {}
    for s in ["train", "val", "test"]:
        with urllib.request.urlopen(SPLIT_URL.format(s), timeout=60) as r:
            pairs[s] = json.load(r)[: CFG["PAIRS"][s]]
    return pairs

def wanted_videos(pairs):
    want = []   # (category, stem, split)
    for split, pl in pairs.items():
        for a, b in pl:
            for vid in (a, b):
                want.append(("Real", vid, split))
            for m in CFG["METHODS"]:
                for x, y in ((a, b), (b, a)):
                    want.append((m, f"{x}_{y}", split))
            if split == "test":
                for x, y in ((a, b), (b, a)):
                    want.append((CFG["UNSEEN_METHOD"], f"{x}_{y}", split))
    return want

def index_zip(zpath):
    idx = {}
    with zipfile.ZipFile(zpath) as zf:
        for info in zf.infolist():
            n = info.filename.replace("\\", "/")
            if info.is_dir() or not n.lower().endswith(".mp4"):
                continue
            parts = n.split("/")
            comps = [p.lower() for p in parts[:-1]]
            stem = os.path.splitext(parts[-1])[0]
            if "actors" in comps:
                continue
            cat = None
            for c in comps:
                if c in CAT_KEYS:
                    cat = CAT_KEYS[c]; break
            if cat is None and any(c in REAL_DIRS for c in comps):
                cat = "Real"
            if cat is None:
                continue
            idx.setdefault((cat, stem), info.filename)
    return idx

def extract_selected(zpath, idx, want, outdir):
    os.makedirs(outdir, exist_ok=True)
    got, missing = [], []
    with zipfile.ZipFile(zpath) as zf:
        for cat, stem, split in tqdm(want, desc="extracting"):
            key = (cat, stem)
            if key not in idx:
                missing.append(key); continue
            dst = f"{outdir}/{cat}__{stem}.mp4"
            if not os.path.exists(dst):
                with zf.open(idx[key]) as s, open(dst, "wb") as f:
                    shutil.copyfileobj(s, f)
            got.append((cat, stem, split, dst))
    return got, missing

In [ ]:
if HAVE_CLIPS:
    print("Resume mode: clips already on Drive -> skipping extraction.")
else:
    PAIRS = load_pairs()
    WANT = wanted_videos(PAIRS)
    IDX = index_zip(ZIP)
    from collections import Counter
    print("videos found in zip per category:", dict(Counter(k[0] for k in IDX)))
    if not any(k[0] == "Real" for k in IDX) or not any(k[0] == "Deepfakes" for k in IDX):
        print("Example paths in zip (paste this output to Claude):")
        with zipfile.ZipFile(ZIP) as zf:
            for n in zf.namelist()[:15]: print("  ", n)
        raise SystemExit("❌ Could not recognise the folder layout inside the zip.")
    GOT, MISSING = extract_selected(ZIP, IDX, WANT, P("videos"))
    print(f"extracted {len(GOT)} videos, missing {len(MISSING)}")
    if MISSING: print("missing examples:", MISSING[:5])
    pd.DataFrame(GOT, columns=["cat", "stem", "split", "path"]).to_csv(P("videos_index.csv"), index=False)

In [ ]:
# free disk: the 10+ GB zip is no longer needed
if "ZIP" in globals() and os.path.exists(ZIP):
    os.remove(ZIP); print("zip deleted")

## Stage 3 — Face-crop clips (8 consecutive frames × 2 clips per video)

In [ ]:
class FaceDetector:
    """MTCNN if it installs cleanly, otherwise OpenCV Haar. Both return (cx, cy, side) or None."""
    def __init__(self):
        self.mtcnn, self.last_src = None, None
        try:
            import torch
            from facenet_pytorch import MTCNN
            self.mtcnn = MTCNN(keep_all=True, device="cuda" if torch.cuda.is_available() else "cpu", post_process=False)
            print("face detector: MTCNN")
        except Exception as e:
            print("MTCNN unavailable -> Haar fallback (", type(e).__name__, ")")
        self.haar = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
        self.stats = {"mtcnn": 0, "haar": 0, "none": 0}

    def _box(self, x1, y1, x2, y2, s):
        w, h = (x2 - x1) / s, (y2 - y1) / s
        return ((x1 + x2) / 2 / s, (y1 + y2) / 2 / s, max(w, h) * 1.3)

    def detect(self, bgr):
        H, W = bgr.shape[:2]
        s = min(1.0, 640.0 / max(H, W))
        small = cv2.resize(bgr, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else bgr
        if self.mtcnn is not None:
            try:
                boxes, _ = self.mtcnn.detect(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
                if boxes is not None and len(boxes):
                    b = max(boxes, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))
                    cx, cy, side = self._box(*b, 1.0)
                    self.last_src = "mtcnn"; self.stats["mtcnn"] += 1
                    return (cx / s, cy / s, side / s)
            except Exception:
                pass
        g = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
        fs = self.haar.detectMultiScale(g, 1.1, 5, minSize=(48, 48))
        if len(fs):
            x, y, w, h = max(fs, key=lambda f: f[2] * f[3])
            cx, cy, side = self._box(x, y, x + w, y + h, 1.0)
            self.last_src = "haar"; self.stats["haar"] += 1
            return (cx / s, cy / s, side / s)
        self.stats["none"] += 1
        return None

def read_clip(path, start, T):
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames = []
    for _ in range(T):
        ok, f = cap.read()
        if not ok: break
        frames.append(f)
    cap.release()
    return frames

def crop_resize(bgr, box, size):
    cx, cy, side = box
    s = int(round(side)); x1 = int(round(cx - side / 2)); y1 = int(round(cy - side / 2))
    x2, y2 = x1 + s, y1 + s
    H, W = bgr.shape[:2]
    pl, pt, pr, pb = max(0, -x1), max(0, -y1), max(0, x2 - W), max(0, y2 - H)
    if pl or pt or pr or pb:
        bgr = cv2.copyMakeBorder(bgr, pt, pb, pl, pr, cv2.BORDER_REPLICATE)
        x1 += pl; x2 += pl; y1 += pt; y2 += pt
    crop = bgr[y1:y2, x1:x2]
    interp = cv2.INTER_AREA if s > size else cv2.INTER_CUBIC
    return cv2.resize(crop, (size, size), interpolation=interp)

def process_video(path, cat, stem, split, det, variant="clean"):
    T, S = CFG["T"], CFG["SIZE"]
    cap = cv2.VideoCapture(path); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    if n < T + 2: return []
    outdir = P("clips", variant); os.makedirs(outdir, exist_ok=True)
    rows = []
    for k, frac in enumerate((0.2, 0.6)[: CFG["CLIPS_PER_VIDEO"]]):
        frames = read_clip(path, int(frac * (n - T)), T)
        if len(frames) < T: continue
        box = det.detect(frames[T // 2])
        if box is None: continue
        crops = np.stack([crop_resize(f, box, S) for f in frames])[..., ::-1]   # BGR -> RGB
        cid = f"{cat}__{stem}__c{k}"
        np.save(f"{outdir}/{cid}.npy", np.ascontiguousarray(crops))
        rows.append(dict(clip_id=cid, video_id=f"{cat}__{stem}", method=cat,
                         label=int(cat != "Real"), split=split, variant=variant))
    return rows

In [ ]:
META_CSV = P("meta_clean.csv")
if os.path.exists(D("checkpoints", "clips_clean.tar")) and not os.path.exists(META_CSV):
    print("restoring clips from Drive checkpoint ...")
    os.makedirs(P("clips"), exist_ok=True)
    !tar -xf {D("checkpoints", "clips_clean.tar")} -C {P("clips")}
    shutil.copy(D("checkpoints", "meta_clean.csv"), META_CSV)

if not os.path.exists(META_CSV):
    det = FaceDetector()
    vids = pd.read_csv(P("videos_index.csv"))
    rows = []
    for r in tqdm(vids.itertuples(), total=len(vids), desc="clips"):
        rows += process_video(r.path, r.cat, r.stem, r.split, det)
    print("detector usage:", det.stats)
    pd.DataFrame(rows).to_csv(META_CSV, index=False)
    # checkpoint to Drive (single big file = fast)
    !tar -cf {P("clips_clean.tar")} -C {P("clips")} clean
    shutil.copy(P("clips_clean.tar"), D("checkpoints", "clips_clean.tar"))
    shutil.copy(META_CSV, D("checkpoints", "meta_clean.csv"))
    os.remove(P("clips_clean.tar"))

META = pd.read_csv(META_CSV)
print(META.groupby(["split", "method"]).size().unstack(0).fillna(0).astype(int))

## Stage 4 — Screen-replay ("played to camera") simulator
Physical model: face shown on an LCD (RGB sub-pixel grid + black matrix) → filmed by a camera at a slight angle
(aliasing = moiré) → lens blur, vignette, banding, color/gamma shift, sensor noise, JPEG.
Three variants are generated from the clean clips:
* `rc_train`  – train+val clips, "seen" screen/camera settings (used for recapture-augmented training)
* `rc_seen`   – test clips, same settings distribution, fresh random draws
* `rc_unseen` – test clips, **different** screens/cameras/compression (tests generalisation)

In [ ]:
PROFILES = {
    "train": dict(k=[4, 5, 6], ppr=(1.15, 1.70), rot=(-3, 3), blur=(0.4, 1.0), noise=(1, 4),
                  gamma=[(0.90, 1.15)], gain_sd=0.03, jpeg=(50, 90), band=(0.0, 0.04), jitter=1.0, vig=(0.05, 0.20)),
    "unseen": dict(k=[7, 8], ppr=(1.70, 2.30), rot=(-6, 6), blur=(0.9, 1.6), noise=(3, 7),
                   gamma=[(0.80, 0.90), (1.20, 1.35)], gain_sd=0.05, jpeg=(30, 55), band=(0.04, 0.09), jitter=1.5, vig=(0.10, 0.30)),
}

def _seed(clip_id, offset):
    return (zlib.crc32(clip_id.encode()) + offset) % (2**32)

def recapture_clip(clip, seed, prof):
    """clip: (T,H,W,3) uint8 RGB -> recaptured clip, same shape."""
    rng = np.random.default_rng(seed)
    T, H, W, _ = clip.shape
    k = int(rng.choice(prof["k"])); ppr = rng.uniform(*prof["ppr"]); rot = rng.uniform(*prof["rot"])
    blur = rng.uniform(*prof["blur"]); noise = rng.uniform(*prof["noise"])
    glo, ghi = prof["gamma"][rng.integers(len(prof["gamma"]))]; gamma = rng.uniform(glo, ghi)
    gains = rng.normal(1.0, prof["gain_sd"], 3).astype(np.float32)
    q = int(rng.uniform(*prof["jpeg"])); vig = rng.uniform(*prof["vig"])
    band_amp = rng.uniform(*prof["band"]); band_per = rng.uniform(30, 90)
    band_ph0 = rng.uniform(0, 2 * np.pi); band_spd = rng.uniform(0.15, 0.6)
    # LCD sub-pixel tile: R,G,B stripes + black matrix
    m = np.zeros((k, k, 3), np.float32)
    for c, ix in enumerate(np.array_split(np.arange(k - 1), 3)):
        m[: k - 1, ix, c] = 1.0
    expo = (1.0 / m.mean(axis=(0, 1))).astype(np.float32)      # camera auto-exposure: compensates the dark pixel matrix
    tile = np.tile(m, (H, W, 1))
    out_px = int(round(H * ppr))
    yy, xx = np.mgrid[0:out_px, 0:out_px].astype(np.float32)
    c0 = (out_px - 1) / 2
    vign = (1.0 - vig * (((xx - c0) ** 2 + (yy - c0) ** 2) / (c0 ** 2)))[..., None]
    outs = []
    for t in range(T):
        up = cv2.resize(clip[t], (W * k, H * k), interpolation=cv2.INTER_NEAREST).astype(np.float32) * tile
        s_int = max(2, int(round(0.75 * k / ppr)))               # camera pixel integrates over its footprint (partial anti-alias -> moire survives)
        up = cv2.boxFilter(up, -1, (s_int, s_int))
        cx, cy = W * k / 2, H * k / 2
        M = cv2.getRotationMatrix2D((cx, cy), rot, ppr / k)
        jx, jy = rng.normal(0, prof["jitter"], 2)
        M[0, 2] += out_px / 2 - cx + jx; M[1, 2] += out_px / 2 - cy + jy
        cam = cv2.warpAffine(up, M, (out_px, out_px), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
        cam = cam * expo * vign
        cam = cam * (1.0 + band_amp * np.sin(2 * np.pi * yy / band_per + band_ph0 + t * band_spd))[..., None]
        cam = 255.0 * np.power(np.clip(cam, 0, 255) / 255.0, gamma) * gains
        cam = cam + rng.normal(0, noise, cam.shape).astype(np.float32)
        cam = cv2.GaussianBlur(cam, (0, 0), blur)
        cam = np.clip(cam, 0, 255).astype(np.uint8)
        cam = cv2.resize(cam, (W, H), interpolation=cv2.INTER_AREA)
        ok, enc = cv2.imencode(".jpg", cam[..., ::-1], [cv2.IMWRITE_JPEG_QUALITY, q])
        outs.append(cv2.imdecode(enc, cv2.IMREAD_COLOR)[..., ::-1])
    return np.ascontiguousarray(np.stack(outs))

def _rc_job(cid, src, dst, seed, prof):
    import cv2 as _cv2; _cv2.setNumThreads(1)
    if os.path.exists(dst): return cid
    np.save(dst, recapture_clip(np.load(src), seed, prof))
    return cid

VARIANTS = {   # name: (profile, splits, seed offset)
    "rc_train":  ("train",  ["train", "val"], 1000),
    "rc_seen":   ("train",  ["test"],         2000),
    "rc_unseen": ("unseen", ["test"],         3000),
}

def make_variants(meta_clean):
    from joblib import Parallel, delayed
    allrows = [meta_clean]
    for vname, (pname, splits, off) in VARIANTS.items():
        sub = meta_clean[meta_clean.split.isin(splits)]
        outdir = P("clips", vname); os.makedirs(outdir, exist_ok=True)
        jobs = [(r.clip_id, P("clips", "clean", r.clip_id + ".npy"), f"{outdir}/{r.clip_id}.npy",
                 _seed(r.clip_id, off), PROFILES[pname]) for r in sub.itertuples()]
        print(f"{vname}: {len(jobs)} clips (profile={pname})")
        Parallel(n_jobs=CFG["NJOBS"], batch_size=4)(delayed(_rc_job)(*j) for j in tqdm(jobs, desc=vname))
        s2 = sub.copy(); s2["variant"] = vname; allrows.append(s2)
    return pd.concat(allrows, ignore_index=True)

In [ ]:
META_ALL_CSV = P("meta_all.csv")
if not os.path.exists(META_ALL_CSV):
    META_ALL = make_variants(pd.read_csv(P("meta_clean.csv")))
    META_ALL.to_csv(META_ALL_CSV, index=False)
META_ALL = pd.read_csv(META_ALL_CSV)
print(META_ALL.groupby("variant").size())

**Figure 1 for your report** — clean vs recaptured examples + their frequency spectra (moiré shows as bright peaks).

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
def load_clip(variant, cid): return np.load(P("clips", variant, cid + ".npy"), mmap_mode="r")

def fig_examples(path):
    m = META_ALL[(META_ALL.split == "test") & (META_ALL.variant == "clean")]
    real = m[m.method == "Real"].iloc[0].clip_id
    fake = m[m.method == "Deepfakes"].iloc[0].clip_id if (m.method == "Deepfakes").any() else m[m.method != "Real"].iloc[0].clip_id
    fig, ax = plt.subplots(2, 6, figsize=(18, 6.2))
    for i, (cid, tag) in enumerate([(real, "REAL"), (fake, "FAKE")]):
        for j, v in enumerate(["clean", "rc_seen", "rc_unseen"]):
            fr = np.array(load_clip(v, cid)[3])
            ax[i, 2 * j].imshow(fr); ax[i, 2 * j].set_title(f"{tag} · {v}"); ax[i, 2 * j].axis("off")
            g = cv2.cvtColor(fr, cv2.COLOR_RGB2GRAY).astype(np.float32)
            F = np.fft.fftshift(np.fft.fft2((g - g.mean()) * np.outer(np.hanning(g.shape[0]), np.hanning(g.shape[1]))))
            ax[i, 2 * j + 1].imshow(np.log10(np.abs(F) ** 2 + 1e-6), cmap="magma"); ax[i, 2 * j + 1].set_title("log power spectrum"); ax[i, 2 * j + 1].axis("off")
    plt.tight_layout(); plt.savefig(path, dpi=130); plt.close()
fig_examples(D("results", "fig1_recapture_examples.png"))
print("saved fig1_recapture_examples.png  (open it from Drive → deepfake_project/results and sanity-check: recaptured faces should look like a photo of a screen)")

## Stage 5 — Hand-crafted image-processing features
* **F** frequency: radial FFT power profile, spectral slope, high-freq ratio, moiré-peak stats, block-DCT band energies, JPEG-blockiness, chroma spectra, banding
* **B** boundary/blending: radial profiles of gradient / noise-residual / Laplacian energy (a face-swap seam shows up as a ring), centre-vs-outer colour & texture ratios
* **T** temporal: frame-difference, optical-flow magnitude / divergence / acceleration (centre vs ring), flicker of luminance, chroma and sharpness

In [ ]:
import scipy.fft
S_ = CFG["SIZE"]; NB = 32; NBR = 12
_yy, _xx = np.indices((S_, S_)); _R = np.hypot(_yy - (S_ - 1) / 2, _xx - (S_ - 1) / 2)
_RB = np.minimum((_R / (S_ / 2) * NB).astype(int), NB)                 # radial bin (NB == outside)
_RHO = _R / (S_ / 2)
_RBB = np.minimum((_RHO * NBR).astype(int), NBR)                       # boundary-ring bins
_HAN = np.outer(np.hanning(S_), np.hanning(S_)).astype(np.float32)
_y1, _x1 = np.indices((112, 112)); _RHO112 = np.hypot(_y1 - 55.5, _x1 - 55.5) / 56.0
_CEN, _RING = _RHO112 < 0.5, (_RHO112 >= 0.5) & (_RHO112 < 0.95)

def _radial(P2, nb, rb):
    m = rb < nb
    return np.bincount(rb[m], P2[m], minlength=nb) / np.maximum(np.bincount(rb[m], minlength=nb), 1)

def _logspec(ch):
    F = np.fft.fftshift(np.fft.fft2((ch - ch.mean()) * _HAN))
    Pw = np.abs(F) ** 2
    return np.log10(Pw + 1e-8), Pw

def _band_peak(v):
    v = v - np.polyval(np.polyfit(np.arange(len(v)), v, 2), np.arange(len(v)))
    s = np.abs(np.fft.rfft(v * np.hanning(len(v))))[2:60]
    return float(s.max() / (s.mean() + 1e-6))

def _dct_feats(ch):
    b = ch.reshape(28, 8, 28, 8).transpose(0, 2, 1, 3).reshape(-1, 8, 8)
    d = scipy.fft.dctn(b - 128.0, axes=(1, 2), norm="ortho")
    L = np.log1p(np.abs(d)).mean(0); s = np.indices((8, 8)).sum(0)
    return [L[s == k].mean() for k in range(1, 15)], float((np.abs(d[:, 4:, 4:]) < 0.5).mean())

def frame_feats(rgb):
    f = {}
    ycc = cv2.cvtColor(rgb, cv2.COLOR_RGB2YCrCb).astype(np.float32)
    Y, Cr, Cb = ycc[..., 0], ycc[..., 1], ycc[..., 2]
    # ---- F: frequency
    P2, Pw = _logspec(Y)
    prof = _radial(P2, NB, _RB)
    for i, v in enumerate(prof): f[f"f_rad{i:02d}"] = v
    f["f_slope"] = np.polyfit(np.log(np.arange(4, NB) + 0.5), prof[4:], 1)[0]
    inside = _R < S_ / 2
    f["f_hf"] = np.log10(Pw[inside & (_R > 0.5 * S_ / 2)].sum() / Pw[inside].sum() + 1e-12)
    exp2d = prof[np.minimum(_RB, NB - 1)]
    msk = (_R >= 0.12 * S_ / 2) & inside
    r = (P2 - exp2d)[msk]; rr = _R[msk]
    f["f_pk_max"] = r.max(); f["f_pk_p999"] = np.percentile(r, 99.9)
    f["f_pk_frac"] = float((r > 3 * r.std()).mean()); f["f_pk_rad"] = rr[r.argmax()] / (S_ / 2)
    dY, zr = _dct_feats(Y)
    for i, v in enumerate(dY): f[f"f_dct{i + 1:02d}"] = v
    f["f_dct_zero"] = zr
    dxx = np.abs(np.diff(Y, axis=1)); dyy = np.abs(np.diff(Y, axis=0))
    f["f_blk_h"] = dxx[:, 7::8].mean() / (dxx.mean() + 1e-6); f["f_blk_v"] = dyy[7::8, :].mean() / (dyy.mean() + 1e-6)
    for nm, ch in (("cr", Cr), ("cb", Cb)):
        pc = _radial(_logspec(ch)[0], 8, np.minimum((_R / (S_ / 2) * 8).astype(int), 8))
        for i, v in enumerate(pc): f[f"f_{nm}_rad{i}"] = v
    f["f_band_row"] = _band_peak(Y.mean(1)); f["f_band_col"] = _band_peak(Y.mean(0))
    # ---- B: boundary / blending
    gx = cv2.Sobel(Y, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(Y, cv2.CV_32F, 0, 1, ksize=3)
    gm = np.hypot(gx, gy)
    nr = np.abs(Y - cv2.GaussianBlur(Y, (0, 0), 1.5)); lp = np.abs(cv2.Laplacian(Y, cv2.CV_32F))
    for nm, mp in (("gm", gm), ("nr", nr), ("lp", lp)):
        pr = _radial(mp, NBR, _RBB)
        for i, v in enumerate(pr): f[f"b_{nm}{i:02d}"] = v
        if nm == "gm": f["b_gm_step"] = float(np.abs(np.diff(pr)).max())
    cen, out = _RHO < 0.35, (_RHO > 0.75) & (_RHO < 1.0)
    f["b_dY"] = Y[cen].mean() - Y[out].mean(); f["b_dCr"] = Cr[cen].mean() - Cr[out].mean(); f["b_dCb"] = Cb[cen].mean() - Cb[out].mean()
    for nm, mp in (("gm", gm), ("nr", nr), ("lp", lp)):
        f[f"b_{nm}_ratio"] = mp[cen].mean() / (mp[out].mean() + 1e-6)
    return f

def temporal_feats(clip):
    T = len(clip)
    g = [cv2.resize(cv2.cvtColor(np.ascontiguousarray(x), cv2.COLOR_RGB2GRAY), (112, 112), interpolation=cv2.INTER_AREA) for x in clip]
    ycc = [cv2.cvtColor(np.ascontiguousarray(x), cv2.COLOR_RGB2YCrCb) for x in clip]
    L = {k: [] for k in ["dc", "dr", "mc", "mr", "ac", "ar", "vc", "vr", "rel"]}
    prev = None
    for t in range(T - 1):
        a, b = g[t], g[t + 1]
        diff = np.abs(b.astype(np.float32) - a.astype(np.float32))
        L["dc"].append(diff[_CEN].mean()); L["dr"].append(diff[_RING].mean())
        fl = cv2.calcOpticalFlowFarneback(a, b, None, 0.5, 2, 15, 2, 5, 1.1, 0)
        mg = np.hypot(fl[..., 0], fl[..., 1])
        L["mc"].append(mg[_CEN].mean()); L["mr"].append(mg[_RING].mean())
        dv = np.abs(np.gradient(fl[..., 0], axis=1) + np.gradient(fl[..., 1], axis=0))
        L["vc"].append(dv[_CEN].mean()); L["vr"].append(dv[_RING].mean())
        L["rel"].append(np.linalg.norm(fl[_CEN].mean(0) - fl[_RING].mean(0)))
        if prev is not None:
            ac = np.hypot(*(fl - prev).transpose(2, 0, 1)); L["ac"].append(ac[_CEN].mean()); L["ar"].append(ac[_RING].mean())
        prev = fl
    m = lambda k: float(np.mean(L[k])); s = lambda k: float(np.std(L[k]))
    f = {"t_diff_c": m("dc"), "t_diff_c_sd": s("dc"), "t_diff_r": m("dr"), "t_diff_r_sd": s("dr"),
         "t_diff_ratio": m("dc") / (m("dr") + 1e-6),
         "t_mag_c": m("mc"), "t_mag_c_sd": s("mc"), "t_mag_r": m("mr"), "t_mag_r_sd": s("mr"), "t_mag_ratio": m("mc") / (m("mr") + 1e-6),
         "t_acc_c": m("ac"), "t_acc_r": m("ar"), "t_acc_ratio": m("ac") / (m("ar") + 1e-6),
         "t_div_c": m("vc"), "t_div_r": m("vr"), "t_div_ratio": m("vc") / (m("vr") + 1e-6),
         "t_rel": m("rel"), "t_rel_sd": s("rel")}
    f["t_lum_sd"] = float(np.std([y[..., 0].mean() for y in ycc]))
    f["t_cr_sd"] = float(np.std([y[..., 1].mean() for y in ycc])); f["t_cb_sd"] = float(np.std([y[..., 2].mean() for y in ycc]))
    lap = [np.abs(cv2.Laplacian(x.astype(np.float32), cv2.CV_32F)).mean() for x in g]
    f["t_lap_cv"] = float(np.std(lap) / (np.mean(lap) + 1e-6))
    return f

def clip_features(clip):
    clip = np.asarray(clip)
    T = len(clip); idx = [0, T // 2, T - 1]
    fr = [frame_feats(np.ascontiguousarray(clip[i])) for i in idx]
    f = {k: float(np.mean([d[k] for d in fr])) for k in fr[0]}
    f.update(temporal_feats(clip))
    return f

def _feat_job(cid, variant, path):
    import cv2 as _cv2; _cv2.setNumThreads(1)
    d = clip_features(np.load(path)); d["clip_id"] = cid; d["variant"] = variant
    return d

def compute_features(meta):
    from joblib import Parallel, delayed
    jobs = [(r.clip_id, r.variant, P("clips", r.variant, r.clip_id + ".npy")) for r in meta.itertuples()]
    out = Parallel(n_jobs=CFG["NJOBS"], batch_size=8)(delayed(_feat_job)(*j) for j in tqdm(jobs, desc="features"))
    return meta.merge(pd.DataFrame(out), on=["clip_id", "variant"], how="left")

In [ ]:
FEAT_CSV = P("features.csv")
if os.path.exists(D("checkpoints", "features.csv")) and not os.path.exists(FEAT_CSV):
    shutil.copy(D("checkpoints", "features.csv"), FEAT_CSV)
if not os.path.exists(FEAT_CSV):
    FEAT = compute_features(META_ALL)
    FEAT.to_csv(FEAT_CSV, index=False); shutil.copy(FEAT_CSV, D("checkpoints", "features.csv"))
FEAT = pd.read_csv(FEAT_CSV)
FCOLS = [c for c in FEAT.columns if c[:2] in ("f_", "b_", "t_")]
GROUPS = {g: [c for c in FCOLS if c.startswith(g.lower() + "_")] for g in "FBT"}
FEAT[FCOLS] = FEAT[FCOLS].replace([np.inf, -np.inf], np.nan)
print({g: len(c) for g, c in GROUPS.items()}, "| rows:", len(FEAT), "| NaNs:", int(FEAT[FCOLS].isna().sum().sum()))

## Stage 6 — Classical (interpretable) detectors
Experiments (all evaluated on the **official test split**, threshold fixed on validation):
* **E1** trained on clean only → tested on clean / recapture-seen / recapture-unseen  *(shows the drop)*
* **E2** trained on clean + simulated recapture → same tests *(shows the recovery)*
* **E3** feature-group ablation (F / B / T / F+B / F+B+T)
* **E4** two-stage pipeline: "is this a screen recapture?" gate → specialist detector *(your Open Question 5)*
* **E5** unseen manipulation (FaceShifter, never trained on)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.inspection import permutation_importance

SUBSETS = {"Deepfakes": ["Deepfakes"], "FaceSwap": ["FaceSwap"], "Face2Face": ["Face2Face"], "NeuralTextures": ["NeuralTextures"],
           "FaceSwap-group": ["Deepfakes", "FaceSwap"], "Reenact-group": ["Face2Face", "NeuralTextures"],
           "ALL-seen": CFG["METHODS"], "FaceShifter(unseen)": [CFG["UNSEEN_METHOD"]]}
TEST_VARIANTS = ["clean", "rc_seen", "rc_unseen"]
KEYCOLS = ["clip_id", "video_id", "method", "label", "variant", "split"]

def _eer(y, p):
    fpr, tpr, _ = roc_curve(y, p); fnr = 1 - tpr; i = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[i] + fnr[i]) / 2)

def metrics(y, p, thr):
    y = np.asarray(y); p = np.asarray(p)
    if len(np.unique(y)) < 2: return dict(AUC=np.nan, EER=np.nan, HTER=np.nan, ACC=np.nan)
    pred = (p >= thr).astype(int)
    far = ((pred == 1) & (y == 0)).sum() / max((y == 0).sum(), 1)   # real rejected as fake
    frr = ((pred == 0) & (y == 1)).sum() / max((y == 1).sum(), 1)   # fake accepted as real
    return dict(AUC=roc_auc_score(y, p), EER=_eer(y, p), HTER=float((far + frr) / 2), ACC=float((pred == y).mean()))

def best_thr(y, p):
    y = np.asarray(y); p = np.asarray(p)
    ths = np.unique(np.quantile(p, np.linspace(0.01, 0.99, 197)))
    def hter(t):
        pred = p >= t
        return (((pred) & (y == 0)).sum() / max((y == 0).sum(), 1) + ((~pred) & (y == 1)).sum() / max((y == 1).sum(), 1)) / 2
    return float(ths[int(np.argmin([hter(t) for t in ths]))])

def eval_table(S, model, thr):
    rows = []
    for v in TEST_VARIANTS:
        Sv = S[(S.variant == v) & (S.split == "test")]
        real = Sv[Sv.method == "Real"]
        for name, ms in SUBSETS.items():
            fk = Sv[Sv.method.isin(ms)]
            if len(fk) == 0 or len(real) == 0: continue
            d = pd.concat([real, fk]); m = metrics(d.label.values, d.prob.values, thr)
            vd = d.groupby("video_id").agg(label=("label", "first"), prob=("prob", "mean"))
            m["AUC_video"] = roc_auc_score(vd.label, vd.prob) if vd.label.nunique() > 1 else np.nan
            rows.append(dict(model=model, variant=v, subset=name, **m))
    return pd.DataFrame(rows)

def train_hgb(df, cols):
    m = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.06, max_leaf_nodes=15, l2_regularization=1.0,
                                       class_weight="balanced", random_state=CFG["SEED"])
    m.fit(df[cols].values, df.label.values); return m

def score(m, df, cols):
    o = df[KEYCOLS].copy(); o["prob"] = m.predict_proba(df[cols].values)[:, 1]; return o

def run_model(name, cols, train_variants, val_variants):
    tr = FEAT[(FEAT.split == "train") & FEAT.variant.isin(train_variants)]
    va = FEAT[(FEAT.split == "val") & FEAT.variant.isin(val_variants)]
    m = train_hgb(tr, cols)
    sv = score(m, va, cols); thr = best_thr(sv.label.values, sv.prob.values)
    st = score(m, FEAT[FEAT.split == "test"], cols)
    return m, eval_table(st, name, thr), st, thr

In [ ]:
RES, SCORES, MODELS = [], {}, {}
ALLC = GROUPS["F"] + GROUPS["B"] + GROUPS["T"]
SETS = {"F": GROUPS["F"], "B": GROUPS["B"], "T": GROUPS["T"], "F+B": GROUPS["F"] + GROUPS["B"], "F+B+T": ALLC}
for tag, tv in (("clean", ["clean"]), ("aug", ["clean", "rc_train"])):
    for gname, cols in SETS.items():
        name = f"HGB[{gname}] train={tag}"
        m, tbl, st, thr = run_model(name, cols, tv, tv)
        RES.append(tbl); SCORES[name] = st
        if gname == "F+B+T": MODELS[tag] = (m, cols, thr)
print("trained", len(SCORES), "classical models")

# ---- E4: recapture gate -> specialist
trg = FEAT[(FEAT.split == "train") & FEAT.variant.isin(["clean", "rc_train"])]
gate = train_hgb(trg.assign(label=(trg.variant != "clean").astype(int)), ALLC)
te = FEAT[FEAT.split == "test"]
pg = gate.predict_proba(te[ALLC].values)[:, 1]
GATE = pd.DataFrame([
    dict(test_set="clean (should be 'no screen')", rate_flagged_as_screen=float((pg[te.variant.values == "clean"] >= .5).mean())),
    dict(test_set="rc_seen", rate_flagged_as_screen=float((pg[te.variant.values == "rc_seen"] >= .5).mean())),
    dict(test_set="rc_unseen", rate_flagged_as_screen=float((pg[te.variant.values == "rc_unseen"] >= .5).mean())),
])
def _gate_auc(variant):
    msk = te.variant.isin(["clean", variant]).values
    return roc_auc_score((te.variant.values[msk] != "clean").astype(int), pg[msk])
GATE["AUC_clean_vs_screen"] = [np.nan, _gate_auc("rc_seen"), _gate_auc("rc_unseen")]
print(GATE.round(3).to_string(index=False))

spec_clean = MODELS["clean"][0]
spec_rc = train_hgb(FEAT[(FEAT.split == "train") & (FEAT.variant == "rc_train")], ALLC)
def pipe_score(df):
    g = gate.predict_proba(df[ALLC].values)[:, 1]
    o = df[KEYCOLS].copy()
    o["prob"] = np.where(g >= .5, spec_rc.predict_proba(df[ALLC].values)[:, 1], spec_clean.predict_proba(df[ALLC].values)[:, 1])
    return o
sv = pipe_score(FEAT[(FEAT.split == "val") & FEAT.variant.isin(["clean", "rc_train"])])
thr_pipe = best_thr(sv.label.values, sv.prob.values)
st = pipe_score(te); SCORES["PIPELINE gate+specialists"] = st
RES.append(eval_table(st, "PIPELINE gate+specialists", thr_pipe))

CLASSICAL = pd.concat(RES, ignore_index=True)
CLASSICAL.to_csv(D("results", "classical_results.csv"), index=False); GATE.to_csv(D("results", "gate_results.csv"), index=False)

def piv(df, models, metric="AUC", subsets=("ALL-seen", "FaceSwap-group", "Reenact-group", "FaceShifter(unseen)")):
    d = df[df.model.isin(models) & df.subset.isin(subsets)]
    return d.pivot_table(index="model", columns=["variant", "subset"], values=metric).reindex(models)
MAIN = ["HGB[F+B+T] train=clean", "HGB[F+B+T] train=aug", "PIPELINE gate+specialists"]
print("\nAUC (higher = better)"); print(piv(CLASSICAL, MAIN).round(3).to_string())
print("\nHTER at val-fixed threshold (lower = better)"); print(piv(CLASSICAL, MAIN, "HTER").round(3).to_string())

**Figures 2–5 for your report** (saved to Drive → `deepfake_project/results/`)

In [ ]:
def fig_spectra(path):
    d = FEAT[(FEAT.split == "train") & FEAT.variant.isin(["clean", "rc_train"])]
    rc = [c for c in FEAT.columns if c.startswith("f_rad")]
    plt.figure(figsize=(7, 4.5))
    for (v, l), g in d.groupby(["variant", "label"]):
        plt.plot(g[rc].mean().values, label=f"{'FAKE' if l else 'REAL'} · {'recaptured' if v != 'clean' else 'clean'}", ls="--" if v != "clean" else "-")
    plt.xlabel("radial frequency bin (low → high)"); plt.ylabel("mean log10 power"); plt.title("Azimuthal-average spectrum (train split)")
    plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig(path, dpi=140); plt.close()

def fig_heat(df, models, path, metric="AUC"):
    fig, ax = plt.subplots(1, 3, figsize=(20, 0.9 + 0.9 * len(models)), sharey=True)
    subs = list(SUBSETS)
    for a, v in zip(ax, TEST_VARIANTS):
        M = np.array([[df[(df.model == m) & (df.variant == v) & (df.subset == s)][metric].mean() for s in subs] for m in models])
        im = a.imshow(M, vmin=0.4, vmax=1.0, cmap="RdYlGn"); a.set_title(f"test: {v}")
        a.set_xticks(range(len(subs))); a.set_xticklabels(subs, rotation=45, ha="right"); a.set_yticks(range(len(models))); a.set_yticklabels(models)
        for i in range(M.shape[0]):
            for j in range(M.shape[1]): a.text(j, i, "" if np.isnan(M[i, j]) else f"{M[i, j]:.2f}", ha="center", va="center", fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.012); plt.suptitle(f"{metric} by attack type and capture condition"); plt.savefig(path, dpi=130, bbox_inches="tight"); plt.close()

def fig_ablation(df, path):
    fig, ax = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    gs = list(SETS)
    for a, v in zip(ax, TEST_VARIANTS):
        for i, tag in enumerate(["clean", "aug"]):
            vals = [df[(df.model == f"HGB[{g}] train={tag}") & (df.variant == v) & (df.subset == "ALL-seen")].AUC.mean() for g in gs]
            a.bar(np.arange(len(gs)) + i * 0.38, vals, 0.38, label=f"train={tag}")
        a.set_xticks(np.arange(len(gs)) + 0.19); a.set_xticklabels(gs); a.set_title(f"test: {v}"); a.set_ylim(0.4, 1.0); a.grid(alpha=.3, axis="y")
    ax[0].set_ylabel("AUC (all 4 seen methods)"); ax[0].legend(); plt.tight_layout(); plt.savefig(path, dpi=140); plt.close()

def fig_roc(scores, names, path, subset="ALL-seen"):
    plt.figure(figsize=(6.5, 6))
    for n in names:
        S = scores[n]
        for v, ls in (("clean", "-"), ("rc_unseen", "--")):
            Sv = S[(S.variant == v) & (S.split == "test")]; d = pd.concat([Sv[Sv.method == "Real"], Sv[Sv.method.isin(SUBSETS[subset])]])
            fpr, tpr, _ = roc_curve(d.label, d.prob); plt.plot(fpr, tpr, ls=ls, label=f"{n} · {v} (AUC {roc_auc_score(d.label, d.prob):.2f})")
    plt.plot([0, 1], [0, 1], "k:", alpha=.4); plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.legend(fontsize=7); plt.grid(alpha=.3)
    plt.title(f"ROC — {subset}"); plt.tight_layout(); plt.savefig(path, dpi=140); plt.close()

def fig_importance(path):
    m, cols, _ = MODELS["aug"]
    va = FEAT[(FEAT.split == "val") & FEAT.variant.isin(["clean", "rc_train"])]
    pi = permutation_importance(m, va[cols].values, va.label.values, scoring="roc_auc", n_repeats=3, random_state=0, n_jobs=1)
    s = pd.Series(pi.importances_mean, index=cols).sort_values(ascending=False).head(20)
    s.to_csv(D("results", "top_features.csv")); plt.figure(figsize=(7, 6)); s[::-1].plot.barh(); plt.xlabel("AUC drop when feature shuffled")
    plt.title("Top-20 features (recapture-augmented model, val)"); plt.tight_layout(); plt.savefig(path, dpi=140); plt.close()
    return s

R = D("results")
fig_spectra(f"{R}/fig2_spectra.png")
fig_heat(CLASSICAL, MAIN, f"{R}/fig3_auc_heatmap_classical.png")
fig_ablation(CLASSICAL, f"{R}/fig4_feature_group_ablation.png")
fig_roc(SCORES, ["HGB[F+B+T] train=clean", "HGB[F+B+T] train=aug"], f"{R}/fig5_roc_classical.png")
TOPF = fig_importance(f"{R}/fig6_top_features.png")
print("figures saved. Top-5 features:", list(TOPF.index[:5]))

In [ ]:
from IPython.display import Image, display
for f in ["fig1_recapture_examples.png", "fig2_spectra.png", "fig3_auc_heatmap_classical.png", "fig4_feature_group_ablation.png"]:
    print(f); display(Image(D("results", f)))

## Stage 7 — Deep baseline (EfficientNet-B0 fine-tune) on the same clips
Two models: `train=clean` and `train=aug` (50% of training frames are the simulated recaptures).

In [ ]:
import torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader
DEV = "cuda" if torch.cuda.is_available() else "cpu"
MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)
BASE = META_ALL[META_ALL.variant == "clean"].reset_index(drop=True)

class FrameDS(Dataset):
    def __init__(self, clips, variants, train, frames=(0, 2, 4, 6), reps=1):
        self.clips = clips.reset_index(drop=True); self.variants = variants; self.train = train
        self.frames = frames; self.reps = reps
        self.n = len(self.clips) * (reps if train else len(frames))
    def __len__(self): return self.n
    def __getitem__(self, i):
        if self.train:
            ci = i % len(self.clips); v = random.choice(self.variants); t = random.randrange(CFG["T"])
        else:
            ci = i // len(self.frames); v = self.variants[0]; t = self.frames[i % len(self.frames)]
        r = self.clips.iloc[ci]
        x = np.asarray(np.load(P("clips", v, r.clip_id + ".npy"), mmap_mode="r")[t], np.float32) / 255.0
        if self.train and random.random() < 0.5: x = x[:, ::-1]
        x = torch.from_numpy(np.ascontiguousarray(((x - MEAN) / STD).transpose(2, 0, 1)))
        return x, torch.tensor(float(r.label)), ci

def _amp(): return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEV == "cuda"))

@torch.no_grad()
def predict(model, clips, variant, bs=128):
    dl = DataLoader(FrameDS(clips, [variant], False), batch_size=bs, num_workers=2)
    model.eval(); L = np.zeros(len(clips)); C = np.zeros(len(clips))
    for x, _, ci in dl:
        with _amp(): o = model(x.to(DEV)).squeeze(1).float().cpu().numpy()
        np.add.at(L, ci.numpy(), o); np.add.at(C, ci.numpy(), 1)
    return 1 / (1 + np.exp(-L / np.maximum(C, 1)))

def clip_scores(model, clips, variant):
    o = clips[KEYCOLS].copy().reset_index(drop=True); o["variant"] = variant; o["prob"] = predict(model, clips, variant); return o

def train_cnn(name, train_variants, epochs=8, reps=6, bs=64, lr=2e-4):
    trc = BASE[BASE.split == "train"]; vac = BASE[BASE.split == "val"]
    dl = DataLoader(FrameDS(trc, train_variants, True, reps=reps), batch_size=bs, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
    model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=1).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(dl), pct_start=0.15)
    pw = torch.tensor([(trc.label == 0).sum() / max((trc.label == 1).sum(), 1)], device=DEV, dtype=torch.float32)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    try: scaler = torch.amp.GradScaler("cuda", enabled=(DEV == "cuda"))
    except Exception: scaler = torch.cuda.amp.GradScaler(enabled=(DEV == "cuda"))
    best, ckpt = -1, P("models", name + ".pt")
    for ep in range(epochs):
        model.train(); tl = 0
        for x, y, _ in tqdm(dl, desc=f"{name} ep{ep + 1}/{epochs}", leave=False):
            x, y = x.to(DEV, non_blocking=True), y.to(DEV)
            with _amp(): out = model(x).squeeze(1)
            loss = crit(out.float(), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); tl += loss.item()
        aucs = []
        for v in train_variants:
            s = clip_scores(model, vac, v); aucs.append(roc_auc_score(s.label, s.prob))
        va = float(np.mean(aucs)); print(f"  ep{ep + 1}: train_loss={tl / len(dl):.3f}  val_AUC={va:.3f}")
        if va > best: best = va; torch.save(model.state_dict(), ckpt)
    model.load_state_dict(torch.load(ckpt, map_location=DEV)); shutil.copy(ckpt, D("checkpoints", name + ".pt"))
    return model

def eval_cnn(model, name, val_variants):
    vac = BASE[BASE.split == "val"]
    sv = pd.concat([clip_scores(model, vac, v) for v in val_variants]); thr = best_thr(sv.label.values, sv.prob.values)
    tec = BASE[BASE.split == "test"]
    st = pd.concat([clip_scores(model, tec, v) for v in TEST_VARIANTS], ignore_index=True)
    return st, eval_table(st, name, thr), thr

CNN_RES, CNN_MODELS = [], {}
for tag, tv in (("clean", ["clean"]), ("aug", ["clean", "rc_train"])):
    name = f"CNN[EffNet-B0] train={tag}"
    mdl = train_cnn(f"effb0_{tag}", tv); CNN_MODELS[tag] = mdl
    st, tbl, thr = eval_cnn(mdl, name, tv); SCORES[name] = st; CNN_RES.append(tbl)
CNN = pd.concat(CNN_RES, ignore_index=True); CNN.to_csv(D("results", "cnn_results.csv"), index=False)
CNN_MAIN = ["CNN[EffNet-B0] train=clean", "CNN[EffNet-B0] train=aug"]
print("\nCNN AUC"); print(piv(CNN, CNN_MAIN).round(3).to_string())
fig_heat(pd.concat([CLASSICAL, CNN]), MAIN + CNN_MAIN, f"{R}/fig7_auc_heatmap_all_models.png")
fig_roc(SCORES, ["CNN[EffNet-B0] train=clean", "CNN[EffNet-B0] train=aug", "HGB[F+B+T] train=aug"], f"{R}/fig8_roc_cnn_vs_classical.png")

In [ ]:
for f in ["fig7_auc_heatmap_all_models.png", "fig8_roc_cnn_vs_classical.png"]:
    print(f); display(Image(D("results", f)))

## Stage 8 — Auto-generated results file for your documentation

In [ ]:
def md_table(df, floatfmt=3):
    df = df.copy(); cols = [" / ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in df.columns]
    idxn = df.index.name or ""; L = ["| " + " | ".join([idxn] + cols) + " |", "|" + "---|" * (len(cols) + 1)]
    for i, r in df.iterrows():
        L.append("| " + " | ".join([str(i)] + [f"{v:.{floatfmt}f}" if isinstance(v, (float, np.floating)) else str(v) for v in r.values]) + " |")
    return "\n".join(L)

ALLRES = CLASSICAL if "CNN" not in globals() else pd.concat([CLASSICAL, CNN], ignore_index=True)
ALLRES.to_csv(D("results", "all_results.csv"), index=False)
MAINALL = MAIN + ([m for m in ["CNN[EffNet-B0] train=clean", "CNN[EffNet-B0] train=aug"] if m in set(ALLRES.model)])
cnt = META_ALL[META_ALL.variant == "clean"].groupby(["method", "split"]).agg(clips=("clip_id", "count"), videos=("video_id", "nunique")).unstack("split").fillna(0).astype(int)
cnt.to_csv(D("results", "dataset_summary.csv"))
md = [f"# Results summary (auto-generated)\n",
      f"Config: {json.dumps({k: v for k, v in CFG.items() if k in ['PAIRS', 'T', 'SIZE', 'CLIPS_PER_VIDEO', 'SEED']})}\n",
      "## Dataset actually used (clips / videos per split)\n", md_table(cnt), "\n",
      "## AUC — main models\n", md_table(piv(ALLRES, MAINALL)), "\n",
      "## HTER at validation-fixed threshold — main models\n", md_table(piv(ALLRES, MAINALL, "HTER")), "\n",
      "## EER — main models\n", md_table(piv(ALLRES, MAINALL, "EER")), "\n",
      "## Video-level AUC (mean over clips) — main models\n", md_table(piv(ALLRES, MAINALL, "AUC_video")), "\n",
      "## Feature-group ablation (AUC, ALL-seen methods)\n",
      md_table(ALLRES[ALLRES.model.str.startswith("HGB[") & (ALLRES.subset == "ALL-seen")].pivot_table(index="model", columns="variant", values="AUC")), "\n",
      "## Recapture gate (E4)\n", md_table(GATE.set_index("test_set")), "\n",
      "## Top features\n", md_table(TOPF.to_frame("perm_importance")), "\n"]
open(D("results", "results_summary.md"), "w").write("\n".join(md))
print("✅ Everything saved to:", D("results"))
print(sorted(os.listdir(D("results"))))

## (Optional, strongest evidence) Stage 9 — Test on REAL screen recaptures you film yourself
The simulated recapture is a proxy. Real footage removes the "it's only simulated" objection. ~30 min of work:
1. Run the next cell → it copies ~45 unseen test videos to your Drive folder `deepfake_project/my_recaptures/source_to_play/`.
2. Download them to your laptop, play them **full-screen on a monitor/phone**, and film the screen with a second phone (face filling most of the frame, ~20–30 s each).
   Tips: film in landscape, steady, face large; iPhone → Settings → Camera → Formats → **Most Compatible** (H.264 instead of HEVC, otherwise OpenCV may not read it).
3. Put your clips in Drive: `my_recaptures/real/` (recordings of the *real* videos) and `my_recaptures/fake/` (recordings of the Deepfakes/NeuralTextures ones), `.mp4` or `.mov`.
4. Run the last cell.

Also worth doing: fill the DeepMoiréFake request form (https://forms.gle/oifqaoujH6q73JnR6) — real recaptured deepfakes from a NeurIPS 2025 dataset; if granted, tell me and I'll add an evaluation cell.

In [ ]:
os.makedirs(D("my_recaptures", "source_to_play"), exist_ok=True); os.makedirs(D("my_recaptures", "real"), exist_ok=True); os.makedirs(D("my_recaptures", "fake"), exist_ok=True)
if not os.path.exists(P("videos_index.csv")):
    raise SystemExit("Source videos are not on this VM (resumed run). Re-run Stage 1–2 once to fetch them, then run this cell.")
vids = pd.read_csv(P("videos_index.csv")); vt = vids[vids.split == "test"]
pick = pd.concat([vt[vt.cat == c].head(15) for c in ["Real", "Deepfakes", "NeuralTextures"]])
for r in pick.itertuples(): shutil.copy(r.path, D("my_recaptures", "source_to_play", os.path.basename(r.path)))
print("copied", len(pick), "videos. File names start with Real__ / Deepfakes__ / NeuralTextures__ so you know which is which.")

In [ ]:
det = FaceDetector(); rows = []
for lab, cat in (("real", "Real"), ("fake", "PhoneFake")):
    for f in sorted(glob.glob(D("my_recaptures", lab, "*"))):
        if f.lower().endswith((".mp4", ".mov", ".m4v", ".avi")):
            rows += process_video(f, cat, os.path.splitext(os.path.basename(f))[0], "custom", det, variant="phone")
PH = pd.DataFrame(rows); print(PH.groupby("method").size())
assert len(PH) > 0 and PH.label.nunique() == 2, "Need both real and fake recordings with a detectable face."
PHF = compute_features(PH)
for tag in ["clean", "aug"]:
    m, cols, thr = MODELS[tag]; p = m.predict_proba(PHF[cols].values)[:, 1]
    mm = metrics(PHF.label.values, p, thr); print(f"HGB[F+B+T] train={tag}: AUC={mm['AUC']:.3f}  HTER={mm['HTER']:.3f}  ACC={mm['ACC']:.3f}")
if "CNN_MODELS" in globals():
    for tag, mdl in CNN_MODELS.items():
        s = clip_scores(mdl, PH, "phone"); mm = metrics(s.label.values, s.prob.values, 0.5)
        print(f"CNN train={tag}: AUC={mm['AUC']:.3f}  ACC@0.5={mm['ACC']:.3f}")
PHF.to_csv(D("results", "real_phone_recapture_features.csv"), index=False)

In [ ]:
# Download everything (tables + figures + summary) to your laptop as one zip
from google.colab import files
shutil.make_archive("/content/results_bundle", "zip", D("results"))
files.download("/content/results_bundle.zip")